# Joint dual outputs: inspect first, train second

The default preview reads the completed HPC smoke run under repo-root `data/`. It needs no full ML dataset and never trains automatically. Set a separate real `DATA_ROOT` only for optional local training.


In [1]:
from pathlib import Path
import runpy
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import resources
from resources.MLdual import DUAL_DATA
from resources.MLmetrics import plot_dual_input_example, write_dual_input_audit

REPO = Path(resources.__file__).resolve().parents[1]
DATA_ROOT = None  # Set a LOCAL parent directory containing MLdata; no SSH here.
RUN_ROOT = REPO / 'data'
PREVIEW_RUN = RUN_ROOT / 'MULTI/Dual/Transformer/dual-MULTI-test-260907'
RUN_LABEL = 'dual-local-review'
NSIMS = 64  # Explicit small-review override; HPC script defaults to all data.
SEED = 42
RUN_TRAINING = False
ALLOW_CPU = False  # Opt in only for a small local debug run.

KeyboardInterrupt: 

## 1. Load a preview

Real-data mode uses the current FCC/Ti profile: 20×19 cells, square cell geometry, Ti/Al pin proportions, and the verified 12-node FT crack subset. The adapter rejects incompatible reference geometry instead of silently inventing BCs. The example profile can handle uniform coordinate rescaling; the inherited DATA body-cropping rules still need scrutiny for differently scaled historical files.

In [ ]:
if DATA_ROOT is None:
    data = None
    audit_dir = PREVIEW_RUN / 'results/input_audit'
    metadata = json.loads((PREVIEW_RUN / 'model_data.json').read_text())
    with np.load(audit_dir / 'input_examples.npz', allow_pickle=False) as file:
        audit = {key: file[key] for key in file.files}
    display(Markdown(f'**Saved HPC inputs:** {PREVIEW_RUN.name}. Only the exported examples are loaded, not training data.'))
else:
    data = DUAL_DATA.from_files(path=str(DATA_ROOT), nsims=NSIMS, split_seed=SEED,
                               range_split=(False, False), load_split=False, save_split=False,
                               LAT='FCC', nnx=20, dis='disNodes', dN=0.2, d_data='in', freq=False,
                               field_input_config={'components': ('U1','U2'), 'drop_frame0': True, 'layout':'auto'})
    display(Markdown(f'Paired splits: **{ {s: len(d) for s,d in data.datasets.items()} }**. No training has run.'))


## 2. Inspect one sample's actual model inputs

Change the sample index and rerun this cell. Grey crosses/red crosses are reference/missing locations, not field predictions. Pin membership comes from the sample's initial disordered position; shared reference channels do not change between UT and FT. In FT the absent rows are excluded from attention and the field loss. UT has an all-present node mask, although missing field frames may still have invalid targets.

In [ ]:
SPLIT = 'train'
SAMPLE_INDEX = 0  # Saved-preview index (0–2), or dataset index when DATA_ROOT is set.
if data is None:
    fig = plot_dual_input_example(audit['canonical_coords'], audit['raw_disorder'][SAMPLE_INDEX],
                                 {m: audit[f'{m}_context'][SAMPLE_INDEX] for m in ('UT','FT')},
                                 {m: audit[f'{m}_node_mask'] for m in ('UT','FT')},
                                 audit['feature_names'].tolist(), metadata['metadata']['context_spec'],
                                 audit['sample_ids'][SAMPLE_INDEX])
else:
    item = data.datasets[SPLIT][SAMPLE_INDEX]
    stats = data.normalizers['geometry']
    raw_delta = item['geometry'].numpy() * stats['scale'].reshape(-1) + stats['mean'].reshape(-1)
    fig = plot_dual_input_example(data.metadata['canonical_coords'], raw_delta,
                                 {m: item['task_features'][m].numpy() for m in ('UT','FT')},
                                 data.node_masks, data.task_feature_names,
                                 data.metadata.get('context_spec', {}), item['sample_id'])
plt.show()


## 3. Export worked arithmetic for the selected sample

Optional: write a labelled plot and short per-node feature tables, including the exact pin-circle calculations, standardisation and presence decisions. The report also records the geometry assumptions. It does not export a full dataset.

In [ ]:
EXPORT_PREVIEW = False
if data is None:
    report = audit_dir / f'sample-{SAMPLE_INDEX+1:02d}.md'
    display(Markdown('\n'.join(line for line in report.read_text().splitlines() if not line.startswith('!['))))
elif EXPORT_PREVIEW:
    report = write_dual_input_audit(data, RUN_ROOT / RUN_LABEL / 'input_preview',
                                   split=SPLIT, sample_indices=[SAMPLE_INDEX])
    display(Markdown(report.read_text()))


## 4. Train using the same entry point as HPC

This deliberately calls the shared single-run entry point rather than copying its optimizer/loss/checkpoint code into the notebook. It reloads real data with the specified run settings; preview illustrations cannot be trained by this cell. Use a new run label to avoid overwriting an existing run. Inspect the four losses separately after training. No HPO, remote operation or Slurm submission occurs here.

In [ ]:
if RUN_TRAINING:
    if DATA_ROOT is None:
        raise ValueError('Set a real LOCAL DATA_ROOT before training.')
    runner = runpy.run_path(str(REPO / 'p2-DisorderML/HPC/DualOutputs/A0-HPC-Dual-test.py'))
    arguments = ['--data-path', str(DATA_ROOT), '--run-root', str(RUN_ROOT),
                 '--run-label', RUN_LABEL, '--nsims', str(NSIMS), '--seed', str(SEED),
                 '--no-range-split', '--epochs', '3', '--batch', '2',
                 '--field-d-model', '128', '--curve-d-model', '128',
                 '--loss', 'combined', '--eval-split', 'val']
    if ALLOW_CPU:
        arguments += ['--allow-cpu']
    model = runner['main'](arguments)
else:
    print('Training disabled. Review the samples and input formulas first.')